# Licensing recommendation analysis

This notebook turns the profile and exploratory analysis into a repeatable recommendation for Kwame Boateng’s fictional $14M back-catalog licensing decision. See the [lab brief](../data/lab_brief.md) and [decision record](../docs/DECISIONS.md).

## Decision questions

1. Does the evidence favor franchise bundles or standalone films, and is that conclusion stable across the required time windows?
2. Which eligible franchises and standalone titles merit consideration in each window, and what supports their inclusion?
3. Do directors or leading-cast members show repeatable associations with stronger outcomes after relevant context is considered?
4. What relative-value case can be made for the strongest franchise, and which KPI most strongly supports it?

## Expected deliverables

- A franchise-versus-standalone comparison reported separately for all years and 2000 onward.
- Side-by-side shortlists targeting three franchises and five standalones per window, with reasons, evidence limits, and rank changes.
- A cautious director and leading-cast signal analysis with role definitions, title counts, and uncertainty.
- A one-paragraph deal note for the strongest franchise and a one-page client memo covering the recommendation, shortlists, both windows, and limitations.

## Interpretation guardrails

Historical box office measures theatrical performance. Popularity changes over time and overlaps with vote count. Genre, talent, and collection patterns are observational. Keep plausible extremes in the primary analysis, and highlight any change in the recommendation between windows.

## What has been done so far

### Data-quality profile — `02_profile_movie_data.ipynb`

- The normalized tables and relationship grains were profiled for the current 112-movie cache.
- No exact raw-object duplicates, exact duplicate movie rows, or title/release-year semantic duplicate candidates were found. Tested range and date checks passed, while the profile also records missing reference labels, constant fields, and narrow language coverage.
- A copy-only dtype experiment preserved the normalized movie frame. Its memory result applies only to this small snapshot; runtime was not benchmarked.

### Exploratory analysis — `03_eda_movie_data.ipynb`

- Numeric distributions, IQR-flagged observations, and plausible extreme values were investigated. Popularity remains snapshot-sensitive.
- Associations were observed between budget and revenue, popularity and vote count, and vote count and rating. These are not causal findings; popularity and vote count overlap.
- Genre, collection, director, and top-billed-cast relationships were explored. Common genre indicators did not improve held-out R-squared over the budget/year/collection controls in this sample. Collection and talent residuals remain descriptive and sample-sensitive.

### Candidate coverage so far

The 127 input rows reconcile to 112 distinct cached titles, 6 repeated rows, 5 TMDB 404s, 3 invalid IDs, and 1 blank. Of the 112 cached titles, 71 were released in 2000 or later. The full pool has 16 titles without a collection ID, and the newer window has 8, before scoring eligibility. See the [findings log](../docs/findings.md).

**Current status**

The profile and EDA are complete for this snapshot. Step 1 rebuilds the current candidate pool and checks both release windows. Step 2 scores the films and produces both shortlists. Step 3 compares the full groups in both windows and checks what happens when each eligible bundle loses its strongest film. The talent analysis and memo follow.

## Implementation plan

### 1. Reconcile inputs and apply the two release windows (implemented below)

- Reconcile every row of the current `id_list.csv` to a cache or fetch outcome. Intersect cached titles with the valid unique IDs in the current list so a previous quarter's cache entries cannot enter the candidate pool. Keep one row per `movie_id`. Report source losses, titles outside the window, and KPI eligibility exclusions separately.
- Use dated, released titles in **A — all years** and only dated, released titles with `release_date >= 2000-01-01` in **B — 2000 onward**. Report missing dates and release statuses. Every summary and score in B uses only its in-window titles. Both windows use the same scoring constants, eligibility gates, and tie-break rules. The windows overlap, so agreement measures sensitivity to the cutoff.
- Report counts by year or decade, title type, and collection, then compare outcome coverage and missingness. Use release year or era for context in nominal financial comparisons. The cutoff addresses Yaw's concern and leaves the historical dollar values nominal.

### 2. Use a fixed audience-first scoring rule (implemented below)

Use vote-adjusted audience rating as the primary shortlist KPI. It measures how TMDB voters rated a title, with fewer-vote ratings pulled toward a common reference. Vote count measures the breadth of that evidence and also serves as the reach-first sensitivity. Retain popularity as context. An audience-first shortlist could favor well-liked films with limited reach, which the reach-first comparison should surface.

- For **title eligibility**, require a finite `vote_average` in [0, 10] and an integer `vote_count` of at least 1,000. Report missing and invalid vote metrics, and titles below the threshold, as separate reasons. Apply financial validity rules to the corresponding KPIs.
- Calculate the **title score** as `(vote_count * vote_average + 1000 * 7.0) / (vote_count + 1000)`. Use 7.0 as a conservative point on the rating scale for this pool, whose exploratory mean is about 7.59. Make observed votes and prior votes count equally at the 1,000-vote entry threshold. These fixed lab assumptions apply across both windows and reruns. Use the full-precision score to rank and round for display.
- Order **standalones** by adjusted rating descending, vote count descending, then numeric `movie_id` ascending. A missing collection ID defines a standalone candidate in the available metadata.
- For **franchise eligibility and ranking**, group by `collection_id` within the window. Allow a collection with one or more observed dated, released films and require a score for each. Flag a collection scored on one film as limited evidence. A known in-window installment with unusable rating evidence makes the proxy bundle ineligible. Rank by median adjusted rating descending, lowest installment adjusted rating descending, median vote count descending, then numeric `collection_id` ascending. Each observed installment gets equal weight. List the observed titles and flag unknown coverage beyond the candidate list. Use these observed titles as the hypothetical lab package.
- For the **shortlists**, take up to three eligible collections and five eligible standalones in order, and explain any shortfall. Calculate positive-budget, positive-gross financial proxies where defined: reported gross minus budget, and `(revenue - budget) / budget`. Log zeros separately from missing values. Show these as context for historical performance, with the valid-film count. TMDB provides no platform outcomes or acquisition costs from which to calculate licensing return. Write a [film exclusions report](../docs/recommendation_exclusions.md) from the same data as the shortlists.

### 3. Compare bundles and standalones at both levels (implemented below)

- Compare like decision units first. Each eligible proxy bundle contributes its median adjusted rating once, and each eligible standalone contributes its adjusted rating once. Report the gap in group medians in rating points, the spread, and the number of eligible units and films. Also show a per-film comparison of all score-eligible collection-linked titles and standalones, so the effect of bundle eligibility and large collections is visible.
- If each side has at least five eligible units, sort decision units by ID within each side. Use NumPy’s default random generator with seed 42 and draw 2,000 bootstrap resamples within each side. Take the 2.5th and 97.5th percentiles of the bundle-minus-standalone median gaps for a descriptive 95% interval. The primary comparison favors bundles if the interval lies entirely above +0.25 rating points, or standalones if it lies entirely below −0.25. Otherwise report insufficient evidence to favor either. The 0.25-point margin and minimum group size are lab judgments. The interval reflects variation within this candidate list.
- Retain plausible extreme values in the primary analysis. Show bundle size, lowest installment score, valid-financial-film counts, and observed gross totals alongside per-film medians. Report the impact of removing the highest-scoring installment from each bundle (lowest numeric movie ID in a score tie) in a sensitivity run, and show which bundles lose their only film or are left with one film.

### 4. Review genre and talent signals

- Count titles per genre and report genre overlaps. Use the EDA cutoffs in both windows. Require at least 10 titles for descriptive comparisons and 15 for genre indicators in an adjusted model. Examine relationships with audience response as well as exploratory log gross. Use budget, release year, genre, and collection status as context where the sample supports it.
- Identify directors by crew job `Director` and use cast order 0–2 as the leading-cast proxy. Deduplicate on `(movie_id, person_id, role)`, group people and collections by ID, and attach names for display. Report distinct titles, distinct collections, and distinct standalone titles. Count each collection as one group and each standalone as its own group. Require at least three distinct titles across at least two groups for a cross-title signal. Flag one-franchise concentrations even when the title threshold is met.
- Compare adjusted outcomes by person and check how the direction changes when the person's largest group by title count is omitted. Break size ties by group type (collection before standalone) and then numeric group ID. If fewer than two titles remain, or the direction flips or disappears, label the signal fragile and show why. Report changes in eligibility and signal between windows. Use release-time and collection-aware holdouts if making any predictive claim. With sparse data, keep the result descriptive and avoid causal language.

### 5. Stress-test and make the rankings reproducible

- Rerun both windows with a reach-first order. Lead with vote count for titles and median vote count for bundles, then use the remaining primary rules in order. Also rerun the primary ranking with vote thresholds of 500 and 5,000 and, separately, with a 6.5 rating anchor. Report who enters, leaves, or moves and whether the leading franchise changes. Recompute the group recommendation for each rating-anchor and vote-threshold sensitivity, and recap the installment-omission result from Step 3. Use nominal per-film gross as a separate comparison with its known era and budget limitations.
- Produce one full candidate table per unit and window, including exclusions and reasons. Show raw and adjusted rating, votes, popularity, financial proxies, year and genre, observed package membership, rank, evidence for inclusion, and the main limitation. Compare shortlist membership and ranks by ID side by side. Label the recommendation and material signals **stable**, **changed**, or **insufficient evidence**.
- Implement the rule once and call it for both windows. Use vectorized checks and aggregations. Record the input and cache fingerprints, code revision, cache timestamp range, parameters, counts at each gate, and elapsed fetch and analysis times separately. Persist terminal fetch failures by ID and reason so Adjoa can distinguish a failed request from a missing TMDB title. Reuse the cached snapshot for an exact rerun. A fresh TMDB fetch can change the evidence. Check ranking ties, missing values, source-list changes, and top-to-bottom execution, then measure a cached run at approximately 500 titles.

### 6. Write the deal note and client memo

- Name the leading eligible lab bundle and its strongest supporting number with the window, denominator, and main limitation. Compare it with the next-best bundle and the strongest standalone alternative. Explain which bundle Kwame should negotiate for first and highlight any change in the leading franchise between windows. Use historical gross and production budget only to describe past theatrical performance. The lab supplies no licensing quotes, so Kwame will need prices to tell what fits within his $14M budget.
- Write one page in business language with the recommendation, both shortlists and their reasons, the two-window comparison, the deal note, and the cost of the remaining unknowns. Point Adjoa to the run record as the proposed improvement beyond the brief. It lets her reproduce a result and explain changes at the next review.

## Step 1 · Which films are in scope?

Read the current candidate CSV, the local cache, and the removal log using the paths in `config.toml`. Run `uv run --locked streamvault` from the project root if the candidate data need to be fetched. Running this notebook reads local files only.

Use the first occurrence of each numeric movie ID. Each film counts once per window. Films dated 1 January 2000 or later can appear once in each window. For an uncached ID, the removal log can show a previous lookup miss for that ID even if its line has moved. An uncached ID with no logged lookup outcome stops the run after the audit table so Adjoa can investigate it.

Each check report shows what was tested and why it passed or failed. A failed check stops the run so an input problem cannot quietly change the recommendation.

In [1]:
import pandas as pd
from IPython.display import display

from streamvault.cache import iter_cached_records
from streamvault.config import CONFIG
from streamvault.diagnostics import raise_for_failed_checks, run_data_checks
from streamvault.load import DUPLICATE_ID_REASON, MovieId, iter_movie_id_rows
from streamvault.normalize import normalize_movie_records

source_path = CONFIG.source_file
# Keep a row for every entry in the CSV, including entries the parser skips.
source_audit = pd.DataFrame.from_records(
    (
        (row.source_line, row.value, None)
        if isinstance(row, MovieId)
        else (row.source_line, row.removed_value, row.reason)
        for row in iter_movie_id_rows(source_path)
    ),
    columns=["source_line", "source_value", "outcome"],
).astype({"source_value": "string", "outcome": "string"})

# Treat IDs like 00123 and 123 as the same film. Keep the first entry.
valid_id = source_audit["outcome"].isna()
source_audit["movie_key"] = source_audit["source_value"].where(valid_id).str.lstrip("0")
duplicate_id = valid_id & source_audit["movie_key"].duplicated()
source_audit.loc[duplicate_id, "outcome"] = DUPLICATE_ID_REASON
current_ids = set(source_audit.loc[source_audit["outcome"].isna(), "movie_key"])


# Read every cache entry so duplicate IDs are visible before we build the film table.
cache_entries = list(iter_cached_records(CONFIG.cache_file))
cached_keys = pd.Index([key.lstrip("0") for key, _ in cache_entries])
in_scope = cached_keys.isin(current_ids)
scoped_keys = cached_keys[in_scope]
scoped_records = [
    record for (_, record), belongs_to_list in zip(cache_entries, in_scope, strict=True)
    if belongs_to_list
]
duplicate_cache_keys = scoped_keys[scoped_keys.duplicated()].unique()
cache_timestamps = pd.to_datetime(
    pd.Series([record.get("cached_at_utc") for record in scoped_records], dtype="object"),
    format="ISO8601",
    errors="coerce",
    utc=True,
)
missing_cache_times = int(cache_timestamps.isna().sum())

load_stage = "After loading the current list and cache"
load_checks = run_data_checks(load_stage, {
    "Source line numbers are unique": (
        bool(source_audit["source_line"].is_unique),
        f"{len(source_audit)} source rows",
    ),
    "Cached movie IDs in scope are unique": (
        len(duplicate_cache_keys) == 0,
        f"{len(duplicate_cache_keys)} repeated IDs. Examples: {duplicate_cache_keys[:5].tolist()}",
    ),
    "Cached films in scope have readable timestamps": (
        missing_cache_times == 0,
        f"{missing_cache_times} missing or unreadable timestamps among {len(scoped_records)} records",
    ),
})
display(load_checks)
raise_for_failed_checks(load_stage, load_checks)

,check,status,details
0,Source line numbers are unique,PASS,127 source rows
1,Cached movie IDs in scope are unique,PASS,0 repeated IDs. Examples: []
2,Cached films in scope have readable timestamps,PASS,0 missing or unreadable timestamps among 112 r...


In [2]:
tables = normalize_movie_records(scoped_records)
movies = tables.movies
movie_ids = movies["movie_id"]
resolved_keys = set(movie_ids.astype("string"))
expected_keys = set(scoped_keys)

clean_stage = "After cleaning the film data"
checks = {
    "Each film has one unique movie ID": (
        bool(movie_ids.notna().all() and movie_ids.is_unique),
        f"{movie_ids.nunique()} distinct IDs across {len(movies)} film rows",
    ),
    "Cleaning keeps exactly the cached films in scope": (
        len(movies) == len(scoped_records) and resolved_keys == expected_keys,
        f"{len(movies)} film rows from {len(scoped_records)} cached entries",
    ),
}

# Missing amounts and zeros stay visible in the coverage report. Impossible amounts stop the run.
for field, label in {"budget": "Production budget", "revenue": "Box office gross"}.items():
    amounts = movies.reindex(columns=[field])[field]
    numeric_amounts = pd.to_numeric(amounts, errors="coerce")
    invalid_amount = amounts.notna() & ~numeric_amounts.between(0, float("inf"), inclusive="left").fillna(False)
    bad_ids = movie_ids.loc[invalid_amount]
    checks[f"{label} is finite and nonnegative when recorded"] = (
        bad_ids.empty,
        f"{len(bad_ids)} invalid values. Example movie IDs: {bad_ids.head(5).tolist()}",
    )

clean_checks = run_data_checks(clean_stage, checks)
display(clean_checks)
raise_for_failed_checks(clean_stage, clean_checks)

,check,status,details
0,Each film has one unique movie ID,PASS,112 distinct IDs across 112 film rows
1,Cleaning keeps exactly the cached films in scope,PASS,112 film rows from 112 cached entries
2,Production budget is finite and nonnegative wh...,PASS,0 invalid values. Example movie IDs: []
3,Box office gross is finite and nonnegative whe...,PASS,0 invalid values. Example movie IDs: []


In [3]:
# Match past lookup misses by ID so they still show up if a row moves.
lookup_reasons = pd.Series(dtype="string")
if CONFIG.removals_file.is_file() and CONFIG.removals_file.stat().st_size:
    removals = pd.read_csv(CONFIG.removals_file, dtype="string", keep_default_na=False)
    lookup_rows = removals.loc[
        removals["source_file"].eq(source_path.name)
        & removals["reason"].str.startswith("TMDB lookup: "),
        ["removed_value", "reason"],
    ].copy()
    lookup_rows["movie_key"] = lookup_rows["removed_value"].str.strip().str.lstrip("0")
    lookup_reasons = lookup_rows.groupby("movie_key")["reason"].agg(
        lambda reasons: " / ".join(sorted(set(reasons)))
    )

# A cached film takes priority over any earlier lookup miss.
pending = source_audit["outcome"].isna()
cached = pending & source_audit["movie_key"].isin(resolved_keys)
uncached = pending & ~cached
unexplained_reason = "No cached record or logged lookup outcome"
source_audit.loc[cached, "outcome"] = "Cached"
source_audit.loc[uncached, "outcome"] = (
    source_audit.loc[uncached, "movie_key"].map(lookup_reasons).fillna(unexplained_reason)
)

source_counts = source_audit["outcome"].value_counts().rename("source_rows")
unexplained_count = int(source_audit["outcome"].eq(unexplained_reason).sum())
source_stage = "After matching the source rows to the cache and removal log"
source_checks = run_data_checks(
    source_stage,
    {
        "Every source row has one outcome": (
            bool(source_audit["source_line"].is_unique and source_audit["outcome"].notna().all()),
            f"{len(source_audit)} source rows accounted for",
        ),
        "The film table preserves the resolved IDs": (
            len(movies) == int(cached.sum()) and resolved_keys == set(source_audit.loc[cached, "movie_key"]),
            f"{len(movies)} distinct films in scope",
        ),
        "Every uncached candidate has a logged outcome": (
            unexplained_count == 0,
            f"{unexplained_count} unexplained misses",
        ),
    },
)

# Show the audit before stopping on a failed check.
display(source_counts.to_frame())
display(source_audit.loc[~cached, ["source_line", "source_value", "outcome"]])
print(f"Cached IDs outside the current list: {cached_keys[~in_scope].nunique()}")
display(source_checks)
raise_for_failed_checks(source_stage, source_checks)

,source_rows
outcome,
Cached,112
Duplicate TMDB movie ID; skipped.,6
TMDB lookup: TMDB returned HTTP 404.,5
Not a positive numeric TMDB movie ID.,3
Blank row; no TMDB movie ID.,1


,source_line,source_value,outcome
16,18,8471293,TMDB lookup: TMDB returned HTTP 404.
28,30,-1,Not a positive numeric TMDB movie ID.
33,35,9412887,TMDB lookup: TMDB returned HTTP 404.
41,43,603,Duplicate TMDB movie ID; skipped.
58,60,278,Duplicate TMDB movie ID; skipped.
67,69,,Blank row; no TMDB movie ID.
71,73,8471294,TMDB lookup: TMDB returned HTTP 404.
88,90,1891,Duplicate TMDB movie ID; skipped.
89,91,155,Duplicate TMDB movie ID; skipped.
91,93,9412888,TMDB lookup: TMDB returned HTTP 404.


Cached IDs outside the current list: 0


,check,status,details
0,Every source row has one outcome,PASS,127 source rows accounted for
1,The film table preserves the resolved IDs,PASS,112 distinct films in scope
2,Every uncached candidate has a logged outcome,PASS,0 unexplained misses


### Release windows

Include a title in all years if it has a date and TMDB marks it as released. The 2000-onward window also requires a release date on or after 1 January 2000. A title without a collection ID is treated as a standalone. Collection counts cover the films observed in this list and show how membership changes at the cutoff.

In [4]:
cutoff = pd.Timestamp("2000-01-01")
candidates = movies.copy()
# Keep date and status columns even when no cached film supplies those fields.
candidates["release_date"] = movies.get(
    "release_date", pd.Series(pd.NaT, index=movies.index, dtype="datetime64[ns]")
)
candidates["status"] = movies.get(
    "status", pd.Series(pd.NA, index=movies.index, dtype="string")
)
candidates["title_type"] = candidates["collection_id"].notna().map(
    {True: "Collection-linked", False: "Standalone"}
)
candidates["release_year"] = candidates["release_date"].dt.year.astype("Int64")
candidates["decade"] = candidates["release_year"] // 10 * 10

dated = candidates["release_date"].notna()
released = candidates["status"].astype("string").eq("Released").fillna(False)
all_years = dated & released
# Use the same release rules in both windows and include 1 January 2000.
since_2000 = all_years & candidates["release_date"].ge(cutoff)

# Use a wide range. The newest cache timestamp keeps its upper end fixed for this snapshot.
first_plausible_year = 1880
snapshot_year = cache_timestamps.max().year
invalid_year = all_years & ~candidates["release_year"].between(
    first_plausible_year, snapshot_year
).fillna(False)
bad_year_ids = candidates.loc[invalid_year, "movie_id"]
year_detail = (
    f"Allowed years: {first_plausible_year} to {snapshot_year}. "
    f"{len(bad_year_ids)} out of range. Example movie IDs: {bad_year_ids.head(5).tolist()}"
    if len(cache_timestamps)
    else "No cached films in scope"
)
windows = {
    "All years": candidates.loc[all_years].copy(),
    "2000 onward": candidates.loc[since_2000].copy(),
}

status_counts = (
    candidates["status"].astype("string").fillna("Missing status")
    .value_counts().rename("films").rename_axis("release_status")
)
display(status_counts.to_frame())
print(f"Films missing a release date: {int((~dated).sum())}")

# Count films with no date first so each excluded film appears under one reason.
window_counts = {}
for window, films in windows.items():
    collection_sizes = films["collection_id"].value_counts()
    window_counts[window] = {
        "Resolved films": len(candidates),
        "Missing release date": int((~dated).sum()),
        "Dated and missing release status": int((dated & candidates["status"].isna()).sum()),
        "Dated and marked with another status": int((dated & candidates["status"].notna() & ~released).sum()),
        "Dated and released but before the cutoff": (
            int((all_years & ~since_2000).sum()) if window == "2000 onward" else 0
        ),
        "Films in window": len(films),
        "Collection-linked films": int(films["collection_id"].notna().sum()),
        "Standalone films": int(films["collection_id"].isna().sum()),
        "Observed collections": len(collection_sizes),
        "Collections with two or more observed films": int(collection_sizes.ge(2).sum()),
    }

display(pd.DataFrame(window_counts))

window_stage = "After applying the release windows"
window_checks = run_data_checks(window_stage, {
    "Dated releases have a plausible year": (bad_year_ids.empty, year_detail),
    "Each window has unique film IDs and accounts for all resolved films": (
        all(films["movie_id"].is_unique for films in windows.values())
        and all(
            counts["Films in window"] + counts["Missing release date"]
            + counts["Dated and missing release status"]
            + counts["Dated and marked with another status"]
            + counts["Dated and released but before the cutoff"] == len(candidates)
            for counts in window_counts.values()
        ),
        f"{len(candidates)} films to account for in each window",
    ),
})
display(window_checks)
raise_for_failed_checks(window_stage, window_checks)

Films missing a release date: 0


,films
release_status,
Released,112


,All years,2000 onward
Resolved films,112,112
Missing release date,0,0
Dated and missing release status,0,0
Dated and marked with another status,0,0
Dated and released but before the cutoff,0,41
Films in window,112,71
Collection-linked films,96,63
Standalone films,16,8
Observed collections,32,24
Collections with two or more observed films,23,17


,check,status,details
0,Dated releases have a plausible year,PASS,Allowed years: 1880 to 2026. 0 out of range. E...
1,Each window has unique film IDs and accounts f...,PASS,112 films to account for in each window


In [5]:
decade_counts = pd.concat(
    {
        window: films.groupby(["decade", "title_type"]).size()
        for window, films in windows.items()
    },
    axis=1,
).fillna(0).astype(int).sort_index()
display(decade_counts)

collection_counts = pd.concat(
    {window: films["collection_id"].value_counts() for window, films in windows.items()},
    axis=1,
).fillna(0).astype(int).sort_index()
# Use the collection ID to attach its name. Names can change or be shared.
collection_ids_before = collection_counts.index
collection_counts = collection_counts.merge(
    tables.collections[["collection_id", "collection_name"]],
    left_index=True,
    right_on="collection_id",
    how="left",
    validate="one_to_one",
    indicator="collection_match",
).set_index("collection_id")

unmatched_collections = collection_counts.index[collection_counts["collection_match"].eq("left_only")]
join_stage = "After joining collection counts to collection names"
join_checks = run_data_checks(join_stage, {
    "The join keeps the same collection rows and IDs": (
        len(collection_counts) == len(collection_ids_before)
        and set(collection_counts.index) == set(collection_ids_before),
        f"{len(collection_ids_before)} rows before the join, {len(collection_counts)} after",
    ),
    "Every observed collection ID matches a reference row": (
        unmatched_collections.empty,
        f"{len(unmatched_collections)} unmatched IDs. Examples: {unmatched_collections[:5].tolist()}",
    ),
})
display(join_checks)
raise_for_failed_checks(join_stage, join_checks)
collection_counts = collection_counts.drop(columns="collection_match")
# The newer window is part of all years, so the difference gives the older count.
collection_counts["Before 2000"] = collection_counts["All years"] - collection_counts["2000 onward"]
cross_cutoff = collection_counts["Before 2000"].gt(0) & collection_counts["2000 onward"].gt(0)
print(f"Collections with observed films on both sides of the cutoff: {int(cross_cutoff.sum())}")
display(collection_counts[["collection_name", "All years", "2000 onward", "Before 2000"]])

All years  2000 onward
decade title_type                               
1960   Standalone                 1            0
1970   Collection-linked          5            0
1980   Collection-linked         12            0
       Standalone                 1            0
1990   Collection-linked         16            0
       Standalone                 6            0
2000   Collection-linked         42           42
       Standalone                 6            6
2010   Collection-linked         21           21
       Standalone                 2            2

,check,status,details
0,The join keeps the same collection rows and IDs,PASS,"32 rows before the join, 32 after"
1,Every observed collection ID matches a referen...,PASS,0 unmatched IDs. Examples: []


Collections with observed films on both sides of the cutoff: 8


,collection_name,All years,2000 onward,Before 2000
collection_id,,,,
10,Star Wars Collection,6,2,4
84,Indiana Jones Collection,4,1,3
119,The Lord of the Rings Collection,3,3,0
230,The Godfather Collection,3,0,3
263,The Dark Knight Collection,3,3,0
264,Back to the Future Collection,3,0,3
295,Pirates of the Caribbean Collection,5,5,0
304,Ocean's Collection,3,3,0
328,Jurassic Park Collection,3,1,2


### Outcome data coverage

Check which films have ratings, votes, popularity, budget, and gross in each window and title type. A reported zero is counted separately from a missing value. Step 2 will apply the scoring and financial validity rules. Missing ratings or votes are listed here so Step 2 can show why each film was excluded. Step 2 must check that every film it actually scores has both inputs.

In [6]:
outcome_fields = ["vote_average", "vote_count", "popularity", "budget", "revenue"]
title_types = ["Collection-linked", "Standalone"]
coverage_tables = {}

for window, films in windows.items():
    # Keep expected fields in the report even when every value is missing.
    outcomes = films.reindex(columns=outcome_fields)
    groups = films["title_type"]
    totals = groups.value_counts().reindex(title_types, fill_value=0)
    missing = outcomes.isna().groupby(groups).sum().reindex(title_types, fill_value=0)
    zeros = outcomes.eq(0).groupby(groups).sum().reindex(title_types, fill_value=0)
    # Show one row per field and title type.
    coverage_tables[window] = pd.concat(
        {
            "Films": pd.DataFrame({field: totals for field in outcome_fields}),
            "Recorded": missing.rsub(totals, axis=0),
            "Missing": missing,
            "Reported zero": zeros,
        },
        axis=1,
    ).stack(level=1)

outcome_coverage = pd.concat(coverage_tables, names=["window", "title_type", "field"])
display(outcome_coverage)

Films  Recorded  Missing  \
window      title_type        field                                    
All years   Collection-linked vote_average     96        96        0   
                              vote_count       96        96        0   
                              popularity       96        96        0   
                              budget           96        96        0   
                              revenue          96        96        0   
            Standalone        vote_average     16        16        0   
                              vote_count       16        16        0   
                              popularity       16        16        0   
                              budget           16        16        0   
                              revenue          16        16        0   
2000 onward Collection-linked vote_average     63        63        0   
                              vote_count       63        63        0   
                              popularity       63        63        0   
                              budget           63        63        0   
                              revenue          63        63        0   
            Standalone        vote_average      8         8        0   
                              vote_count        8         8        0   
                              popularity        8         8        0   
                              budget            8         8        0   
                              revenue           8         8        0   

                                            Reported zero  
window      title_type        field                        
All years   Collection-linked vote_average              0  
                              vote_count                0  
                              popularity                0  
                              budget                    0  
                              revenue                   0  
            Standalone        vote_average              0  
                              vote_count                0  
                              popularity                0  
                              budget                    0  
                              revenue                   0  
2000 onward Collection-linked vote_average              0  
                              vote_count                0  
                              popularity                0  
                              budget                    0  
                              revenue                   0  
            Standalone        vote_average              0  
                              vote_count                0  
                              popularity                0  
                              budget                    0  
                              revenue                   0

### Step 1 conclusion

**Step 1 checks passed.** The checks passed at loading, cleaning, source matching, window selection, and the collection join. Every film in both windows has a recorded rating and vote count.

- **The candidate list is accounted for.** Of 127 rows, 112 gave us distinct films, 6 were repeats, 5 returned a “not found” response from TMDB, 3 had invalid IDs, and 1 was blank. All 112 films have release dates and are marked as released. We have no film evidence for the nine rows lost to lookup or input problems, so the shortlist can only reflect the films we resolved.
- **The 2000 cutoff changes the choices.** It removes 41 films and halves the standalone pool from 16 to 8. Filling five standalone spots would use five of those eight before any scoring exclusions. Collections with at least two films in the list fall from 23 to 17. Eight collections have films on both sides of the cutoff. For example, Indiana Jones goes from four observed films to one, so the newer window scores it using one film.
- **Inputs for Step 2.** Ratings, vote counts, popularity, production budget, and reported box office gross are present and nonzero for every film in both windows. We can now check the values, apply the 1,000-vote minimum, and rank the eligible titles and collections in each window. The franchise-versus-standalone comparison follows in Step 3.

## Step 2 · Which films and bundles make the shortlists?

Use the same rating adjustment and vote minimum in both windows. A rating of 9.0 with 1,000 votes becomes 8.0 because the film's votes and the 7.0 reference have equal weight at that point. Votes above the minimum give the film's own rating more weight. Keep the full score for ranking and round the displayed values.

The score reports why a film could not be scored. If any in-window installment cannot be scored, its collection cannot qualify as a bundle. A franchise can qualify with one film, and we label the score as evidence from that one film. The studio package may contain other installments.

In [7]:
minimum_votes = 1_000
anchor_rating = 7.0
anchor_votes = 1_000
bundle_target = 3
standalone_target = 5

def score_films(films, *, minimum_votes, anchor_rating, anchor_votes):
    # Preserve the film rows and make missing columns visible as missing values.
    scored = films.reindex(columns=films.columns.union(
        ["title", *outcome_fields], sort=False
    )).copy()
    raw_rating = scored["vote_average"]
    raw_votes = scored["vote_count"]
    rating = pd.to_numeric(raw_rating, errors="coerce")
    votes = pd.to_numeric(raw_votes, errors="coerce")
    # A true/false value can turn into 1/0 when converted to a number.
    valid_rating = rating.between(0, 10).fillna(False) & ~raw_rating.astype("string").isin(
        ["True", "False"]
    )
    finite_votes = votes.between(0, float("inf"), inclusive="left").fillna(False)
    valid_votes = (
        votes.where(finite_votes).mod(1).eq(0).fillna(False)
        & ~raw_votes.astype("string").isin(["True", "False"])
    )
    enough_votes = votes.ge(minimum_votes).fillna(False)
    eligible = valid_rating & valid_votes & enough_votes

    rating_problem = pd.Series("", index=films.index, dtype="string")
    rating_problem.loc[~valid_rating] = "Invalid rating"
    rating_problem.loc[raw_rating.isna()] = "Missing rating"
    vote_problem = pd.Series("", index=films.index, dtype="string")
    vote_problem.loc[valid_votes & ~enough_votes] = f"Fewer than {minimum_votes:,} votes"
    vote_problem.loc[~valid_votes] = "Invalid vote count"
    vote_problem.loc[raw_votes.isna()] = "Missing vote count"
    scored["score_exclusion"] = rating_problem.str.cat(vote_problem, sep=". ").str.strip(". ")

    # A weighted average keeps the adjusted score on the same 0 to 10 scale.
    scored["vote_average"] = rating.where(valid_rating)
    scored["vote_count"] = votes.where(valid_votes)
    eligible_votes = votes.where(eligible).astype(float)
    scored["adjusted_rating"] = (
        eligible_votes * rating.where(eligible) + anchor_votes * anchor_rating
    ) / (eligible_votes + anchor_votes)

    # Zero and missing inputs are counted in Step 1. Leave the dollar comparisons blank for both.
    budget = pd.to_numeric(scored["budget"], errors="coerce").astype(float)
    gross = pd.to_numeric(scored["revenue"], errors="coerce").astype(float)
    # A film can count toward observed gross even when its budget is missing.
    has_gross = gross.between(0, float("inf"), inclusive="neither")
    has_financials = budget.between(0, float("inf"), inclusive="neither") & has_gross
    scored["reported_gross"] = gross.where(has_gross)
    scored["gross_minus_budget"] = gross.where(has_financials) - budget.where(has_financials)
    scored["gross_difference_to_budget"] = scored["gross_minus_budget"] / budget.where(has_financials)
    # Use the same set of film rows when comparing gross and budget within a bundle.
    scored["gross_on_valid_pair"] = gross.where(has_financials)
    scored["budget_on_valid_pair"] = budget.where(has_financials)
    return scored


scored_windows = {
    window: score_films(
        films, minimum_votes=minimum_votes, anchor_rating=anchor_rating, anchor_votes=anchor_votes
    )
    for window, films in windows.items()
}

score_stage = "After scoring films"
score_checks = {}
for window, scored in scored_windows.items():
    # Check that every score has the evidence it needs and every unscored film has a reason.
    has_score = scored["adjusted_rating"].notna()
    votes = scored["vote_count"].where(has_score)
    valid_score = (
        scored["vote_average"].between(0, 10).fillna(False)
        & votes.between(minimum_votes, float("inf"), inclusive="left").fillna(False)
        & votes.mod(1).eq(0).fillna(False)
        & scored["adjusted_rating"].between(0, 10).fillna(False)
    )
    reasons = scored["score_exclusion"]
    wrong_reason = reasons.isna() | has_score.ne(reasons.eq("").fillna(False))
    bad_ids = scored.loc[(has_score & ~valid_score) | wrong_reason, "movie_id"]
    score_checks[f"{window}: Scores have valid inputs and exclusions have reasons"] = (
        bad_ids.empty,
        f"{int(has_score.sum())} scored, {int((~has_score).sum())} unscored. "
        f"{len(bad_ids)} issues. Example movie IDs: {bad_ids.head(5).tolist()}",
    )
score_report = run_data_checks(score_stage, score_checks)
raise_for_failed_checks(score_stage, score_report)

In [8]:
def rank_candidates(candidates, order):
    # The last sort key is a numeric ID, so a tie never depends on input order.
    ranked = candidates.sort_values(order, ascending=[False] * (len(order) - 1) + [True])
    return ranked.assign(rank=range(1, len(ranked) + 1))


def summarize_bundles(scored, collection_names):
    # Count every in-window installment, including any without a usable score.
    linked = scored.loc[scored["collection_id"].notna()]
    bundles = linked.groupby("collection_id").agg(
        films=("movie_id", "size"),
        scored_films=("adjusted_rating", "count"),
        median_adjusted_rating=("adjusted_rating", "median"),
        lowest_adjusted_rating=("adjusted_rating", "min"),
        median_votes=("vote_count", "median"),
        films_with_gross=("reported_gross", "count"),
        total_reported_gross=("reported_gross", "sum"),
        median_reported_gross=("reported_gross", "median"),
        films_with_budget_and_gross=("gross_minus_budget", "count"),
        median_budget=("budget_on_valid_pair", "median"),
        median_gross=("gross_on_valid_pair", "median"),
        median_gross_minus_budget=("gross_minus_budget", "median"),
        median_gross_difference_to_budget=("gross_difference_to_budget", "median"),
    ).reset_index()
    # A blank total means we have no reported gross for any film in the bundle.
    bundles["total_reported_gross"] = bundles["total_reported_gross"].where(bundles["films_with_gross"].gt(0))
    bundles["collection_name"] = bundles["collection_id"].map(collection_names)
    bundles["film_evidence"] = "Multiple films"
    bundles.loc[bundles["films"].eq(1), "film_evidence"] = "Based on one film"
    return bundles


collection_names = collection_counts["collection_name"]
bundle_summaries = {}
ranked_bundles = {}
ranked_standalones = {}
bundle_shortlists = {}
standalone_shortlists = {}
shortlisted_bundle_films = {}

for window, scored in scored_windows.items():
    bundles = summarize_bundles(scored, collection_names)
    bundle_summaries[window] = bundles
    # Each group contains at least one observed film. Every film in the group must have a score.
    eligible_bundles = bundles.loc[bundles["scored_films"].eq(bundles["films"])]
    ranked_bundles[window] = rank_candidates(eligible_bundles, [
        "median_adjusted_rating", "lowest_adjusted_rating", "median_votes", "collection_id",
    ])

    standalone = scored["collection_id"].isna()
    eligible_standalones = scored.loc[standalone & scored["adjusted_rating"].notna()]
    ranked_standalones[window] = rank_candidates(eligible_standalones, [
        "adjusted_rating", "vote_count", "movie_id",
    ])
    bundle_shortlists[window] = ranked_bundles[window].head(bundle_target)
    standalone_shortlists[window] = ranked_standalones[window].head(standalone_target)

    # Attach a single bundle rank to each member by ID. A name can belong to multiple collections.
    shortlist_ranks = bundle_shortlists[window].set_index("collection_id")["rank"]
    member_rank = scored["collection_id"].map(shortlist_ranks)
    members = scored.loc[member_rank.notna()].assign(bundle_rank=member_rank.dropna().astype(int))
    shortlisted_bundle_films[window] = members.sort_values(["bundle_rank", "release_date", "movie_id"])

shortlist_stage = "After ranking candidates and attaching films to shortlisted bundles"
shortlist_checks = {}
for window, members in shortlisted_bundle_films.items():
    shortlist = bundle_shortlists[window]
    expected_member_ids = scored_windows[window].loc[
        scored_windows[window]["collection_id"].isin(shortlist["collection_id"]), "movie_id"
    ]
    shortlist_checks[f"{window}: Each shortlisted bundle keeps all its scored films"] = (
        members["movie_id"].is_unique
        and set(members["movie_id"]) == set(expected_member_ids)
        and len(members) == int(shortlist["films"].sum())
        and bool(shortlist["films"].ge(1).all() and members["adjusted_rating"].notna().all()),
        f"{len(members)} member rows, {len(expected_member_ids)} expected, for {len(shortlist)} bundles",
    )
shortlist_report = run_data_checks(shortlist_stage, shortlist_checks)
raise_for_failed_checks(shortlist_stage, shortlist_report)

### Shortlists for both windows

A bundle's median score is the middle score of its films, or the average of the two middle scores. The lowest score shows how its weakest observed installment rated. With one observed film, the median and lowest score are the same. For a standalone, use the film's own adjusted score. See the member tables for exactly which films count toward each bundle.

The financial columns show what these films did at the box office in historical dollars. The difference between gross and production budget is shown in dollars and as a share of budget. Bundle figures are medians across films with both amounts positive, and the tables give that count. Kwame would need licensing quotes to judge what fits his $14M budget.

In [9]:
from IPython.display import Markdown

# Rename and round for display only. The ranked tables keep full precision.
film_columns = {
    "movie_id": "Movie ID", "title": "Film", "release_year": "Year",
    "vote_average": "Rating", "vote_count": "Votes", "adjusted_rating": "Adjusted rating",
    "popularity": "Popularity", "budget": "Budget ($)", "revenue": "Gross ($)",
    "gross_minus_budget": "Gross less budget ($)",
    "gross_difference_to_budget": "Difference / budget",
}
bundle_columns = {
    "rank": "Rank", "collection_id": "Collection ID", "collection_name": "Bundle", "films": "Films",
    "film_evidence": "Evidence", "median_adjusted_rating": "Median adjusted rating", "lowest_adjusted_rating": "Lowest adjusted rating",
    "median_votes": "Median votes", "films_with_budget_and_gross": "Films with positive budget and gross",
    "median_budget": "Median budget ($)", "median_gross": "Median gross ($)",
    "median_gross_minus_budget": "Median gross less budget ($)",
    "median_gross_difference_to_budget": "Median difference / budget",
}
for window in windows:
    bundle_count = len(ranked_bundles[window])
    standalone_count = len(ranked_standalones[window])
    display(Markdown(
        f"### {window}\n\n{bundle_count} eligible bundles and {standalone_count} eligible standalones."
    ))
    if bundle_count < bundle_target:
        print(f"Only {bundle_count} of {bundle_target} bundle spots can be filled under the eligibility rules.")
    if standalone_count < standalone_target:
        print(f"Only {standalone_count} of {standalone_target} standalone spots can be filled under the eligibility rules.")

    display(Markdown(
        "#### Franchise shortlist\n\n"
        "The top bundles, ranked by the middle adjusted rating of their films. "
        "The Evidence column flags scores based on one film."
    ))
    display(bundle_shortlists[window][list(bundle_columns)].rename(columns=bundle_columns).round(3))
    member_columns = {"bundle_rank": "Bundle rank", **film_columns}
    display(Markdown(
        "#### Films behind the bundle scores\n\n"
        "The films that count toward each shortlisted bundle in this release window. "
        "Bundle rank links each film to the table above."
    ))
    display(shortlisted_bundle_films[window][list(member_columns)].rename(columns=member_columns).round(3))
    standalone_columns = {"rank": "Rank", **film_columns}
    display(Markdown(
        "#### Standalone shortlist\n\n"
        "The top individual films, ranked by their own adjusted rating."
    ))
    display(standalone_shortlists[window][list(standalone_columns)].rename(columns=standalone_columns).round(3))

### All years

32 eligible bundles and 16 eligible standalones.

#### Franchise shortlist

The top bundles, ranked by the middle adjusted rating of their films. The Evidence column flags scores based on one film.

,Rank,Collection ID,Bundle,Films,Evidence,Median adjusted rating,Lowest adjusted rating,Median votes,Films with positive budget and gross,Median budget ($),Median gross ($),Median gross less budget ($),Median difference / budget
3,1,230,The Godfather Collection,3,Multiple films,8.472,7.362,14626.0,3,13000000.0,136766062.0,89600000.0,6.892
2,2,119,The Lord of the Rings Collection,3,Multiple films,8.402,8.378,27846.0,3,93000000.0,926287400.0,847287400.0,10.725
20,3,9743,The Hannibal Lecter Collection,1,Based on one film,8.278,8.278,18863.0,1,19000000.0,272742922.0,253742922.0,13.355


#### Films behind the bundle scores

The films that count toward each shortlisted bundle in this release window. Bundle rank links each film to the table above.

,Bundle rank,Movie ID,Film,Year,Rating,Votes,Adjusted rating,Popularity,Budget ($),Gross ($),Gross less budget ($),Difference / budget
59,1,238,The Godfather,1972,8.688,23999,8.620,40.742,6000000,245066411,2.390664e+08,39.844
60,1,240,The Godfather Part II,1974,8.573,14626,8.472,24.044,13000000,102600000,8.960000e+07,6.892
2,1,242,The Godfather Part III,1990,7.412,7199,7.362,13.736,54000000,136766062,8.276606e+07,1.533
18,2,120,The Lord of the Rings: The Fellowship of the Ring,2001,8.451,28890,8.402,39.398,93000000,871368364,7.783684e+08,8.370
16,2,121,The Lord of the Rings: The Two Towers,2002,8.433,25000,8.378,28.702,79000000,926287400,8.472874e+08,10.725
33,2,122,The Lord of the Rings: The Return of the King,2003,8.507,27846,8.455,34.941,94000000,1118888979,1.024889e+09,10.903
42,3,274,The Silence of the Lambs,1991,8.346,18863,8.278,5.329,19000000,272742922,2.537429e+08,13.355


#### Standalone shortlist

The top individual films, ranked by their own adjusted rating.

... (4 outputs left out)

#### Films behind the bundle scores

The films that count toward each shortlisted bundle in this release window. Bundle rank links each film to the table above.

,Bundle rank,Movie ID,Film,Year,Rating,Votes,Adjusted rating,Popularity,Budget ($),Gross ($),Gross less budget ($),Difference / budget
18,1,120,The Lord of the Rings: The Fellowship of the Ring,2001,8.451,28890,8.402,39.398,93000000,871368364,7.783684e+08,8.370
16,1,121,The Lord of the Rings: The Two Towers,2002,8.433,25000,8.378,28.702,79000000,926287400,8.472874e+08,10.725
33,1,122,The Lord of the Rings: The Return of the King,2003,8.507,27846,8.455,34.941,94000000,1118888979,1.024889e+09,10.903
29,2,24428,The Avengers,2012,8.085,40389,8.059,42.105,220000000,1518815515,1.298816e+09,5.904
84,2,99861,Avengers: Age of Ultron,2015,7.278,25366,7.267,33.755,235000000,1405403694,1.170404e+09,4.980
51,2,299536,Avengers: Infinity War,2018,8.247,33418,8.211,49.466,300000000,2052415039,1.752415e+09,5.841
27,2,299534,Avengers: Endgame,2019,8.247,29384,8.206,89.118,356000000,2942649778,2.586650e+09,7.266
22,3,24,Kill Bill: Vol. 1,2003,7.970,19739,7.923,14.684,30000000,180906076,1.509061e+08,5.030
99,3,393,Kill Bill: Vol. 2,2004,7.872,15687,7.820,11.554,30000000,152159461,1.221595e+08,4.072


#### Standalone shortlist

The top individual films, ranked by their own adjusted rating.

,Rank,Movie ID,Film,Year,Rating,Votes,Adjusted rating,Popularity,Budget ($),Gross ($),Gross less budget ($),Difference / budget
63,1,129,Spirited Away,2001,8.537,19231,8.461,37.198,19000000,274925095,255925095.0,13.470
108,2,157336,Interstellar,2014,8.493,41925,8.458,56.357,165000000,746606706,581606706.0,3.525
37,3,27205,Inception,2010,8.400,40754,8.366,45.414,160000000,839030630,679030630.0,4.244
50,4,1124,The Prestige,2006,8.212,18478,8.150,17.847,40000000,109676311,69676311.0,1.742
75,5,77,Memento,2000,8.200,17054,8.134,15.472,9000000,40060108,31060108.0,3.451


### Which films were left out?

This step writes the [film exclusions report](../docs/recommendation_exclusions.md) each time the notebook runs. It covers the source rows lost before scoring, plus every resolved film that missed either shortlist. A film can appear on one shortlist and miss the other. For a film with a rank, the report explains whether its standalone rank or its bundle's rank kept it off the list.

In [10]:
from html import escape

# Give each resolved film one outcome per window. The release reasons follow the Step 1 audit order.
# Leave eligible releases blank until they get a scoring or ranking outcome.
release_outcome = pd.Series(pd.NA, index=candidates.index, dtype="string")
release_outcome.loc[~released] = "Release status is " + candidates.loc[~released, "status"].astype("string")
release_outcome.loc[candidates["status"].isna()] = "Missing release status"
release_outcome.loc[~dated] = "Missing release date"
on_shortlist = "Shortlisted"
film_outcomes = candidates.reindex(columns=["movie_id", "title"]).copy()
for window, scored in scored_windows.items():
    bundles = bundle_summaries[window].set_index("collection_id")
    bundle_rank = scored["collection_id"].map(ranked_bundles[window].set_index("collection_id")["rank"])
    standalone_rank = scored["movie_id"].map(ranked_standalones[window].set_index("movie_id")["rank"])
    # Give a film's own scoring problem first. Otherwise explain its bundle or rank.
    reason = scored["score_exclusion"].copy()
    linked = scored["collection_id"].notna()
    good_score = reason.eq("")
    reason.loc[good_score & linked & bundle_rank.isna()] = "Another in-window bundle film could not be scored"
    reason.loc[bundle_rank.gt(bundle_target)] = (
        "Bundle ranked #" + bundle_rank.dropna().astype(int).astype("string")
        + f" (top {bundle_target} selected)"
    )
    one_film_below_cut = bundle_rank.gt(bundle_target) & scored["collection_id"].map(bundles["films"]).eq(1)
    reason.loc[one_film_below_cut] = reason.loc[one_film_below_cut] + ". Based on one film"
    reason.loc[standalone_rank.gt(standalone_target)] = (
        "Standalone ranked #" + standalone_rank.dropna().astype(int).astype("string")
        + f" (top {standalone_target} selected)"
    )
    reason.loc[bundle_rank.le(bundle_target) | standalone_rank.le(standalone_target)] = on_shortlist

    window_outcome = release_outcome.copy()
    if window == "2000 onward":
        window_outcome.loc[all_years & ~since_2000] = "Before 1 January 2000"
    window_outcome.loc[scored.index] = reason
    film_outcomes[window] = window_outcome

exclusion_stage = "Before writing the film exclusions report"
exclusion_checks = {}
for window in windows:
    outcome = film_outcomes[window]
    accounted_ids = set(film_outcomes.loc[outcome.eq(on_shortlist), "movie_id"])
    shortlisted_ids = set(shortlisted_bundle_films[window]["movie_id"]) | set(
        standalone_shortlists[window]["movie_id"]
    )
    exclusion_checks[f"{window}: Every film has an outcome that agrees with the shortlist"] = (
        bool(outcome.notna().all() and outcome.ne("").all() and accounted_ids == shortlisted_ids),
        f"{len(accounted_ids)} shortlisted films and {len(outcome) - len(accounted_ids)} films left out",
    )
exclusion_report = run_data_checks(exclusion_stage, exclusion_checks)
raise_for_failed_checks(exclusion_stage, exclusion_report)
# Show one summary of the Stage 2 checks after verifying each step.
display(Markdown(
    "#### Scoring and shortlist checks\n\n"
    "These checks verify the scores, the films counted in each bundle, "
    "and the outcome recorded for every film."
))
display(pd.concat([score_report, shortlist_report, exclusion_report], ignore_index=True))


def markdown_table(frame):
    # Format only for the report. Pipes and newlines in a title should stay inside its table cell.
    def clean_cell(value):
        return escape(" ".join(str(value).splitlines())).replace("|", "\\|")

    header = "| " + " | ".join(frame.columns) + " |"
    divider = "| " + " | ".join(["---"] * len(frame.columns)) + " |"
    rows = [
        "| " + " | ".join(clean_cell(value) for value in row) + " |"
        for row in frame.itertuples(index=False, name=None)
    ]
    return "\n".join([header, divider, *rows])


source_exclusions = source_audit.loc[
    source_audit["outcome"].ne("Cached"), ["source_line", "source_value", "outcome"]
].rename(columns={"source_line": "CSV line", "source_value": "Source value", "outcome": "Reason"})
source_exclusions["Source value"] = source_exclusions["Source value"].fillna("").replace("", "(blank)")

film_exclusions = film_outcomes.loc[film_outcomes[list(windows)].ne(on_shortlist).any(axis=1)].copy()
film_exclusions["title"] = film_exclusions["title"].fillna("(title unavailable)")
film_exclusions = film_exclusions.sort_values(["title", "movie_id"]).rename(
    columns={"movie_id": "Movie ID", "title": "Film"}
)
summary = (
    film_outcomes[list(windows)].eq(on_shortlist).agg(["sum", "size"]).T
    .rename(columns={"sum": "Shortlisted films", "size": "Resolved films"})
    .rename_axis("Window").reset_index()
)
summary["Films left out"] = summary["Resolved films"] - summary["Shortlisted films"]

report_path = CONFIG.project_root / "docs" / "recommendation_exclusions.md"
report_path.write_text("\n".join([
    "# Recommendation exclusions", "",
    "Generated by [Stage 2 of the recommendation notebook](../notebooks/04_recommendation.ipynb).",
    "Run the notebook from the top to refresh this report for a new candidate list or cache.", "",
    "## Summary", "",
    "Each film in a shortlisted bundle counts as one shortlisted film in this report.", "",
    markdown_table(summary), "",
    "## Source rows left out before scoring", "",
    f"{len(source_exclusions)} of {len(source_audit)} CSV rows were left out here.",
    "A repeat row has the same movie ID as an earlier entry. Unknown titles are recorded by their source value.",
    "",
    markdown_table(source_exclusions) if len(source_exclusions) else "No source rows were left out.",
    "",
    "## Resolved films left out in at least one window", "",
    "A rank in this table is the film or bundle's position among eligible candidates.",
    "A film shortlisted in both windows is omitted from this table.", "",
    markdown_table(film_exclusions) if len(film_exclusions) else "No resolved films were left out.", "",
]), encoding="utf-8")
print(f"Wrote {report_path.relative_to(CONFIG.project_root).as_posix()}")

Wrote docs/recommendation_exclusions.md


#### Scoring and shortlist checks

These checks verify the scores, the films counted in each bundle, and the outcome recorded for every film.

,check,status,details
0,All years: Scores have valid inputs and exclus...,PASS,"112 scored, 0 unscored. 0 issues. Example movi..."
1,2000 onward: Scores have valid inputs and excl...,PASS,"71 scored, 0 unscored. 0 issues. Example movie..."
2,All years: Each shortlisted bundle keeps all i...,PASS,"7 member rows, 7 expected, for 3 bundles"
3,2000 onward: Each shortlisted bundle keeps all...,PASS,"9 member rows, 9 expected, for 3 bundles"
4,All years: Every film has an outcome that agre...,PASS,12 shortlisted films and 100 films left out
5,2000 onward: Every film has an outcome that ag...,PASS,14 shortlisted films and 98 films left out


### Step 2 conclusion for the current snapshot

- **Both windows filled all shortlist spots.** Every film in each window passed the rating and vote checks. All years had 32 eligible bundles and 16 eligible standalones. From 2000 onward there were 24 bundles and 8 standalones. Allowing one-film evidence added nine eligible bundles across all years and seven from 2000 onward.
- **The leading bundle changed.** The Godfather leads across all years with a median adjusted rating of 8.47. The Lord of the Rings leads from 2000 onward at 8.40 and has the strongest lowest film score of the shortlisted bundles in either window (8.38). The Hannibal Lecter Collection takes third place across all years based on one film, The Silence of the Lambs. The Avengers and Kill Bill take second and third from 2000 onward.
- **The leading standalone also changed.** The Shawshank Redemption leads across all years at 8.68. Spirited Away leads from 2000 onward at 8.46. Spirited Away and Interstellar are the two standalones on both lists.
- **Every film has a recorded outcome.** The [exclusions report](../docs/recommendation_exclusions.md) accounts for all 15 CSV rows left out before scoring and gives a reason for each resolved film that missed either shortlist.

**Step 3 uses the full groups of eligible bundles and standalones to compare the two options.**

## Step 3 · Which type of option do the ratings favor?

Use every eligible option from Step 2. Each franchise contributes the middle adjusted rating of its observed films. Each standalone contributes its own adjusted rating. The gap is the bundle median minus the standalone median. A positive gap favors bundles on this measure. A negative gap favors standalones on this measure.

Show the middle half of scores using the 25th and 75th percentiles. When both groups have at least five options, repeatedly draw an equal-sized sample from each group with replacement. An option can appear more than once in a draw. Keep the middle 95% of the 2,000 resulting gaps. The choice uses that range and a 0.25-point margin. The range describes variation within this candidate list. The sample and margin rules are analyst choices.

Then count each eligible film separately. This lets us see how the picture changes when each installment has its own weight, including films whose bundles could not qualify as a whole.

In [11]:
import numpy as np

comparison_minimum = 5
comparison_margin = 0.25
bootstrap_draws = 2_000
bootstrap_seed = 42
insufficient_evidence = "Insufficient evidence to favor either group"


def describe_rating_gap(bundle_scores, standalone_scores):
    # Use the same summary when each row is a bundle and when each row is a film.
    bundle_quartiles = bundle_scores.quantile([0.25, 0.50, 0.75])
    standalone_quartiles = standalone_scores.quantile([0.25, 0.50, 0.75])
    return {
        "bundle_units": len(bundle_scores),
        "standalone_units": len(standalone_scores),
        "bundle_median": bundle_quartiles.loc[0.50],
        "bundle_q1": bundle_quartiles.loc[0.25],
        "bundle_q3": bundle_quartiles.loc[0.75],
        "standalone_median": standalone_quartiles.loc[0.50],
        "standalone_q1": standalone_quartiles.loc[0.25],
        "standalone_q3": standalone_quartiles.loc[0.75],
        "rating_gap": bundle_quartiles.loc[0.50] - standalone_quartiles.loc[0.50],
    }


def compare_decision_units(bundles, standalones):
    # Sorting by ID keeps the same seed tied to the same options if rows are reordered.
    bundle_scores = bundles.sort_values("collection_id")["median_adjusted_rating"]
    standalone_scores = standalones.sort_values("movie_id")["adjusted_rating"]
    result = describe_rating_gap(bundle_scores, standalone_scores)
    result.update(
        bundle_films=int(bundles["films"].sum()),
        interval_low=np.nan,
        interval_high=np.nan,
        conclusion=insufficient_evidence,
    )
    if min(len(bundle_scores), len(standalone_scores)) < comparison_minimum:
        return result

    # Each draw uses the original group size. Only the films or bundles drawn can repeat.
    rng = np.random.default_rng(bootstrap_seed)
    bundle_samples = rng.choice(bundle_scores.to_numpy(dtype=float), size=(bootstrap_draws, len(bundle_scores)))
    standalone_samples = rng.choice(
        standalone_scores.to_numpy(dtype=float), size=(bootstrap_draws, len(standalone_scores)),
    )
    gaps = np.median(bundle_samples, axis=1) - np.median(standalone_samples, axis=1)
    low, high = np.quantile(gaps, [0.025, 0.975])
    result.update(interval_low=low, interval_high=high)
    if low > comparison_margin:
        result["conclusion"] = "Favors bundles"
    elif high < -comparison_margin:
        result["conclusion"] = "Favors standalones"
    return result


decision_rows = []
film_rows = []
comparison_checks = {}
for window, scored in scored_windows.items():
    bundles = ranked_bundles[window]
    standalones = ranked_standalones[window]
    has_score = scored["adjusted_rating"].notna()
    linked = scored["collection_id"].notna()

    # A collection with any unscored installment stays out of the main comparison.
    ineligible_ids = scored.loc[linked & ~has_score, "collection_id"]
    eligible_members = scored.loc[linked & ~scored["collection_id"].isin(ineligible_ids)]
    eligible_standalone_ids = scored.loc[~linked & has_score, "movie_id"]
    comparison_checks[f"{window}: The comparison has one row per eligible option"] = (
        bundles["collection_id"].is_unique
        and standalones["movie_id"].is_unique
        and set(bundles["collection_id"]) == set(eligible_members["collection_id"])
        and int(bundles["films"].sum()) == len(eligible_members)
        and set(standalones["movie_id"]) == set(eligible_standalone_ids),
        f"{len(bundles)} bundles covering {len(eligible_members)} films, {len(standalones)} standalones",
    )
    decision_rows.append({"window": window, **compare_decision_units(bundles, standalones)})
    film_rows.append({
        "window": window,
        **describe_rating_gap(
            scored.loc[linked & has_score, "adjusted_rating"],
            scored.loc[~linked & has_score, "adjusted_rating"],
        ),
    })

decision_comparison = pd.DataFrame(decision_rows).set_index("window")
film_comparison = pd.DataFrame(film_rows).set_index("window")

In [12]:
def omit_best_bundle_films(scored, eligible_bundles):
    # Start with the eligible bundles so this check cannot add new options.
    members = scored.loc[scored["collection_id"].isin(eligible_bundles["collection_id"])]
    # Use the lowest numeric movie ID if two films share the highest score.
    removed = members.sort_values(
        ["adjusted_rating", "movie_id"], ascending=[False, True],
    ).drop_duplicates("collection_id")
    remaining = members.loc[~members["movie_id"].isin(removed["movie_id"])]
    return removed, summarize_bundles(remaining, collection_names)


omission_rows = []
omission_details = {}
for window, scored in scored_windows.items():
    original = ranked_bundles[window].set_index("collection_id")
    removed, reduced = omit_best_bundle_films(scored, ranked_bundles[window])
    # Keep a row even when removing the only observed film makes the bundle disappear.
    detail = original[["collection_name", "films", "median_adjusted_rating"]].join(
        removed.set_index("collection_id")[["movie_id", "title", "adjusted_rating"]].rename(columns={
            "movie_id": "removed_movie_id", "title": "removed_title", "adjusted_rating": "removed_score",
        }),
        validate="one_to_one",
    ).join(
        reduced.set_index("collection_id")[["films", "median_adjusted_rating"]].rename(columns={
            "films": "films_left", "median_adjusted_rating": "new_median",
        }),
        validate="one_to_one",
    )
    detail["films_left"] = detail["films_left"].fillna(0).astype(int)
    detail["median_change"] = detail["new_median"] - detail["median_adjusted_rating"]
    omission_details[window] = detail
    comparison_checks[f"{window}: The sensitivity removed one film from each eligible bundle"] = (
        detail["removed_movie_id"].notna().all()
        and detail["removed_movie_id"].is_unique
        and detail["films_left"].eq(detail["films"] - 1).all(),
        f"{len(removed)} films removed from {len(original)} bundles",
    )
    omission_rows.append({
        "window": window,
        **compare_decision_units(reduced, ranked_standalones[window]),
        "bundles_lost": int(detail["films_left"].eq(0).sum()),
        "bundles_left_with_one": int(detail["films_left"].eq(1).sum()),
    })

omission_comparison = pd.DataFrame(omission_rows).set_index("window")
omission_comparison["gap_change"] = omission_comparison["rating_gap"] - decision_comparison["rating_gap"]

comparison_stage = "After preparing the bundle and standalone comparisons"
comparison_report = run_data_checks(comparison_stage, comparison_checks)
raise_for_failed_checks(comparison_stage, comparison_report)

In [13]:
# Use descriptive headings and round only the displayed numbers.
spread_columns = {
    "bundle_median": "Bundle median", "bundle_q1": "Bundle 25th pct.", "bundle_q3": "Bundle 75th pct.",
    "standalone_median": "Standalone median", "standalone_q1": "Standalone 25th pct.",
    "standalone_q3": "Standalone 75th pct.", "rating_gap": "Gap (points)",
}
interval_columns = {
    "interval_low": "95% range low", "interval_high": "95% range high", "conclusion": "Ratings conclusion",
}
decision_columns = {
    "bundle_units": "Eligible bundles", "bundle_films": "Films in eligible bundles",
    "standalone_units": "Standalone films", **spread_columns, **interval_columns,
}
display(Markdown(
    "### Each bundle and standalone counts once\n\n"
    "This is the main comparison. The 25th and 75th percentiles mark the middle half of each group's scores. "
    f"The 95% range is blank if a group has fewer than {comparison_minimum} options. "
    f"The whole range must put one group ahead by more than {comparison_margin:.2f} points to favor it."
))
display(decision_comparison[list(decision_columns)].rename(columns=decision_columns).rename_axis("Window").round(3))
if decision_comparison["conclusion"].nunique() > 1:
    display(Markdown("**The ratings conclusion changes when the 2000 cutoff is applied.**"))
else:
    display(Markdown("**The ratings conclusion is the same in both windows.**"))

film_columns_to_show = {
    "bundle_units": "Scored collection films", "standalone_units": "Scored standalone films", **spread_columns,
    "bundle_median": "Collection films median", "bundle_q1": "Collection films 25th pct.",
    "bundle_q3": "Collection films 75th pct.",
}
display(Markdown(
    "### Each eligible film counts once\n\n"
    "Every scored film gets equal weight here, so a larger franchise contributes more scores. "
    "The gap compares the middle collection-film and standalone-film scores."
))
display(film_comparison[list(film_columns_to_show)].rename(columns=film_columns_to_show).rename_axis("Window").round(3))

### Each bundle and standalone counts once

This is the main comparison. The 25th and 75th percentiles mark the middle half of each group's scores. The 95% range is blank if a group has fewer than 5 options. The whole range must put one group ahead by more than 0.25 points to favor it.

,Eligible bundles,Films in eligible bundles,Standalone films,Bundle median,Bundle 25th pct.,Bundle 75th pct.,Standalone median,Standalone 25th pct.,Standalone 75th pct.,Gap (points),95% range low,95% range high,Ratings conclusion
Window,,,,,,,,,,,,,
All years,32,96,16,7.586,7.068,7.783,8.360,8.124,8.440,-0.774,-1.149,-0.491,Favors standalones
2000 onward,24,63,8,7.150,6.795,7.705,8.142,8.091,8.389,-0.991,-1.454,-0.497,Favors standalones


**The ratings conclusion is the same in both windows.**

### Each eligible film counts once

Every scored film gets equal weight here, so a larger franchise contributes more scores. The gap compares the middle collection-film and standalone-film scores.

,Scored collection films,Scored standalone films,Collection films median,Collection films 25th pct.,Collection films 75th pct.,Standalone median,Standalone 25th pct.,Standalone 75th pct.,Gap (points)
Window,,,,,,,,,
All years,96,16,7.564,6.957,7.877,8.360,8.124,8.440,-0.796
2000 onward,63,8,7.404,6.788,7.771,8.142,8.091,8.389,-0.738


In [14]:
evidence_columns = {
    "collection_name": "Bundle", "collection_id": "Collection ID", "films": "Observed films",
    "scored_films": "Scored films", "median_adjusted_rating": "Median film score",
    "lowest_adjusted_rating": "Lowest film score", "films_with_gross": "Films with gross",
    "total_reported_gross": "Total gross ($)", "median_reported_gross": "Median film gross ($)",
    "films_with_budget_and_gross": "Films with budget and gross", "median_budget": "Median paired budget ($)",
    "median_gross_minus_budget": "Median paired gross less budget ($)",
}

for window, bundles in bundle_summaries.items():
    display(Markdown(
        f"### {window} · What goes into each bundle\n\n"
        "These are the observed films behind each bundle's scores and box office figures. "
        "Gross totals and medians use films with positive reported gross. Paired figures use films "
        "with both positive gross and budget. The dollar figures are historical and use the amounts as reported."
    ))
    shown = bundles.sort_values(
        ["median_adjusted_rating", "collection_id"], ascending=[False, True],
    )[list(evidence_columns)].rename(columns=evidence_columns)
    with pd.option_context("display.max_rows", None):
        display(shown.round(3).reset_index(drop=True))

### All years · What goes into each bundle

These are the observed films behind each bundle's scores and box office figures. Gross totals and medians use films with positive reported gross. Paired figures use films with both positive gross and budget. The dollar figures are historical and use the amounts as reported.

,Bundle,Collection ID,Observed films,Scored films,Median film score,Lowest film score,Films with gross,Total gross ($),Median film gross ($),Films with budget and gross,Median paired budget ($),Median paired gross less budget ($)
0,The Godfather Collection,230,3,3,8.472,7.362,3,4.844325e+08,1.367661e+08,3,13000000.0,8.960000e+07
1,The Lord of the Rings Collection,119,3,3,8.402,8.378,3,2.916545e+09,9.262874e+08,3,93000000.0,8.472874e+08
2,The Hannibal Lecter Collection,9743,1,1,8.278,8.278,1,2.727429e+08,2.727429e+08,1,19000000.0,2.537429e+08
3,The Lion King Collection,94032,1,1,8.192,8.192,1,7.634556e+08,7.634556e+08,1,45000000.0,7.184556e+08
4,The Avengers Collection,86311,4,4,8.132,7.267,4,7.919284e+09,1.785615e+09,4,267500000.0,1.525615e+09
5,Blade Runner Collection,422837,1,1,7.881,7.881,1,4.172242e+07,4.172242e+07,1,28000000.0,1.372242e+07
6,Kill Bill Collection,2883,2,2,7.871,7.820,2,3.330655e+08,1.665328e+08,2,30000000.0,1.365328e+08
7,"Monsters, Inc. Collection",137696,1,1,7.820,7.820,1,6.500000e+08,6.500000e+08,1,130000000.0,5.200000e+08
8,The Dark Knight Collection,263,3,3,7.771,7.703,3,2.463740e+09,1.008481e+09,3,185000000.0,8.234805e+08
9,Harry Potter Collection,1241,8,8,7.735,7.643,8,7.759778e+09,9.360860e+08,8,140000000.0,7.613048e+08


### 2000 onward · What goes into each bundle

These are the observed films behind each bundle's scores and box office figures. Gross totals and medians use films with positive reported gross. Paired figures use films with both positive gross and budget. The dollar figures are historical and use the amounts as reported.

,Bundle,Collection ID,Observed films,Scored films,Median film score,Lowest film score,Films with gross,Total gross ($),Median film gross ($),Films with budget and gross,Median paired budget ($),Median paired gross less budget ($)
0,The Lord of the Rings Collection,119,3,3,8.402,8.378,3,2.916545e+09,9.262874e+08,3,93000000.0,8.472874e+08
1,The Avengers Collection,86311,4,4,8.132,7.267,4,7.919284e+09,1.785615e+09,4,267500000.0,1.525615e+09
2,Kill Bill Collection,2883,2,2,7.871,7.820,2,3.330655e+08,1.665328e+08,2,30000000.0,1.365328e+08
3,"Monsters, Inc. Collection",137696,1,1,7.820,7.820,1,6.500000e+08,6.500000e+08,1,130000000.0,5.200000e+08
4,The Dark Knight Collection,263,3,3,7.771,7.703,3,2.463740e+09,1.008481e+09,3,185000000.0,8.234805e+08
5,Harry Potter Collection,1241,8,8,7.735,7.643,8,7.759778e+09,9.360860e+08,8,140000000.0,7.613048e+08
6,The Incredibles Collection,468222,1,1,7.694,7.694,1,6.314421e+08,6.314421e+08,1,92000000.0,5.394421e+08
7,Mad Max Collection,8945,1,1,7.617,7.617,1,3.788583e+08,3.788583e+08,1,150000000.0,2.288583e+08
8,Toy Story Collection,10194,2,2,7.596,7.421,2,2.141157e+09,1.070579e+09,2,187500000.0,8.830787e+08
9,The Bourne Collection,31562,3,3,7.404,7.304,3,9.453586e+08,2.885002e+08,3,70000000.0,2.135002e+08


In [15]:
omission_columns = {
    "bundle_units": "Bundles left", "bundle_films": "Bundle films left",
    "standalone_units": "Standalone films", "bundles_lost": "Bundles that disappear",
    "bundles_left_with_one": "Bundles left with one film",
    "bundle_median": "Bundle median", "standalone_median": "Standalone median",
    "rating_gap": "Gap (points)", "gap_change": "Change in gap (points)", **interval_columns,
}
display(Markdown(
    "### How much does each bundle depend on its strongest film?\n\n"
    "Repeat the main comparison after removing the highest-scoring film from each eligible bundle. "
    "The standalone group stays the same. Change in gap is the new gap minus the original gap."
))
display(omission_comparison[list(omission_columns)].rename(columns=omission_columns).rename_axis("Window").round(3))

detail_columns = {
    "collection_name": "Bundle", "collection_id": "Collection ID", "removed_title": "Film removed",
    "films": "Films before", "films_left": "Films left", "median_adjusted_rating": "Old median",
    "new_median": "New median", "median_change": "Change in median (points)",
}
limited_bundles = pd.concat(omission_details, names=["Window", "collection_id"]).reset_index()
limited_bundles = limited_bundles.loc[limited_bundles["films_left"].le(1)]
display(Markdown(
    "### Which bundles disappear or are left with one film?\n\n"
    "A bundle with no films left drops out of this check. A bundle with one film left still counts and "
    "uses that film's adjusted rating."
))
if limited_bundles.empty:
    display(Markdown("No eligible bundles fell below two films in this run."))
else:
    shown = limited_bundles[["Window", *detail_columns]].rename(columns=detail_columns)
    with pd.option_context("display.max_rows", None):
        display(shown.round(3).reset_index(drop=True))

display(Markdown(
    "### Comparison checks\n\n"
    "These checks verify the options counted in the main comparison and the films removed in the sensitivity run."
))
display(comparison_report)

### How much does each bundle depend on its strongest film?

Repeat the main comparison after removing the highest-scoring film from each eligible bundle. The standalone group stays the same. Change in gap is the new gap minus the original gap.

,Bundles left,Bundle films left,Standalone films,Bundles that disappear,Bundles left with one film,Bundle median,Standalone median,Gap (points),Change in gap (points),95% range low,95% range high,Ratings conclusion
Window,,,,,,,,,,,,
All years,23,64,16,9,1,7.282,8.360,-1.078,-0.304,-1.477,-0.732,Favors standalones
2000 onward,17,39,8,7,5,6.977,8.142,-1.165,-0.173,-1.585,-0.647,Favors standalones


### Which bundles disappear or are left with one film?

A bundle with no films left drops out of this check. A bundle with one film left still counts and uses that film's adjusted rating.

,Window,Bundle,Collection ID,Film removed,Films before,Films left,Old median,New median,Change in median (points)
0,All years,The Hannibal Lecter Collection,9743,The Silence of the Lambs,1,0,8.278,NaN,NaN
1,All years,The Lion King Collection,94032,The Lion King,1,0,8.192,NaN,NaN
2,All years,Blade Runner Collection,422837,Blade Runner,1,0,7.881,NaN,NaN
3,All years,Kill Bill Collection,2883,Kill Bill: Vol. 1,2,1,7.871,7.820,-0.052
4,All years,"Monsters, Inc. Collection",137696,"Monsters, Inc.",1,0,7.820,NaN,NaN
5,All years,The Incredibles Collection,468222,The Incredibles,1,0,7.694,NaN,NaN
6,All years,Jaws Collection,2366,Jaws,1,0,7.634,NaN,NaN
7,All years,Mad Max Collection,8945,Mad Max: Fury Road,1,0,7.617,NaN,NaN
8,All years,Ghostbusters Collection,2980,Ghostbusters,1,0,7.452,NaN,NaN
9,All years,The Smurfs (Theatrical) Collection,134897,The Smurfs,1,0,5.995,NaN,NaN


### Comparison checks

These checks verify the options counted in the main comparison and the films removed in the sensitivity run.

,check,status,details
0,All years: The comparison has one row per elig...,PASS,"32 bundles covering 96 films, 16 standalones"
1,2000 onward: The comparison has one row per el...,PASS,"24 bundles covering 63 films, 8 standalones"
2,All years: The sensitivity removed one film fr...,PASS,32 films removed from 32 bundles
3,2000 onward: The sensitivity removed one film ...,PASS,24 films removed from 24 bundles


### Step 3 conclusion for the current snapshot

- **The ratings favor standalones in both windows.** Across all years, the middle standalone scored **8.36** versus **7.59** for the middle bundle. The standalone lead was **0.77 points**, with a resampled range of **0.49 to 1.15**. From 2000 onward, the middle scores were **8.14** and **7.15**, a standalone lead of **0.99 points** with a range of **0.50 to 1.45**. Both ranges clear the 0.25-point margin. The date cutoff changes the size of the gap while leaving the direction the same.
- **The same group leads when we count every film separately.** Standalones lead by **0.80 points** across all years and **0.74** from 2000 onward. Giving larger bundles more weight through their extra films narrows the newer-window gap.
- **Removing the best film from each bundle leaves standalones ahead.** The lead rises to **1.08 points** across all years and **1.16** from 2000 onward. The check removes **9** and **7** bundles that had only one observed film, respectively. Another **1** and **5** bundles, respectively, are left with one film. The wider gap reflects both weaker film lineups and the loss of those one-film bundles.
- **Box office adds context.** The Avengers' four observed films reported **$7.92B** in total gross, with a median of **$1.79B** per film. Harry Potter's eight reported **$7.76B** in total, with a median of **$936M**. Both figures describe past theatrical performance. Checking what fits Kwame's $14M requires licensing quotes.

**Ready for Step 4.** For this candidate list, the ratings give Kwame a reason to prioritize standalones overall while keeping the top franchise targets on the shortlist. Next, check whether any director, lead actor, or genre shows a pattern across titles.